# Day 6: Data pipeline (solution)

Reference solutions. Every cell is runnable; `practice.ipynb` is the blank-page twin.

## `s1_data.py`: Data — stability, transpose, set algebra, hashability.

In [ ]:
rows = [("a", 3), ("b", 1), ("c", 3)]

In [ ]:
assert sorted(rows, key=lambda r: r[1]) == [("b", 1), ("a", 3), ("c", 3)]  # stable

In [ ]:
names, nums = zip(*rows)

In [ ]:
assert names == ("a", "b", "c")

In [ ]:
assert {1, 2, 3} & {2, 3, 4} == {2, 3} and {1, 2} - {2} == {1} and {1} ^ {2} == {1, 2}
d = {(1, 2): "pt"}

In [ ]:
assert d[(1, 2)] == "pt"

In [ ]:
try: d[[1]] = 1
except TypeError: pass
else: raise AssertionError
assert list(range(10))[::3] == [0, 3, 6, 9]

**Then, vary it:**

- VARIATION: sort by two keys with a tuple key and reverse one using negation
- VARIATION: dedupe a list preserving order with dict.fromkeys

## `s2_flow.py`: Flow — chained generators, takewhile, send().

In [ ]:
import itertools
def nums(): yield from range(10)
def evens(it): return (x for x in it if x % 2 == 0)
def squares(it): return (x * x for x in it)

In [ ]:
import itertools

assert list(squares(evens(nums()))) == [0, 4, 16, 36, 64]
assert sum(itertools.takewhile(lambda x: x < 20, squares(nums()))) == 30

In [ ]:
def running_avg():
    total = n = 0
    while True:
        x = yield (total / n if n else 0)
        total += x
        n += 1
g = running_avg()
next(g)

In [ ]:
assert g.send(10) == 10 and g.send(20) == 15

**Then, vary it:**

- VARIATION: add a stage that drops multiples of 3; nothing runs until you consume
- VARIATION: replace nums() with itertools.count() and still take 5 results

## `s3_functions.py`: Functions — compose, itemgetter, partial in map.

In [ ]:
from functools import partial, reduce
from operator import itemgetter
def compose(*fs): return reduce(lambda f, g: lambda x: g(f(x)), fs)
inc, dbl = (lambda x: x + 1), (lambda x: x * 2)

In [ ]:
assert compose(inc, dbl)(3) == 8

In [ ]:
rows = [{"n": "a", "v": 2}, {"n": "b", "v": 1}]

In [ ]:
from operator import itemgetter

assert [r["n"] for r in sorted(rows, key=itemgetter("v"))] == ["b", "a"]

In [ ]:
assert list(map(partial(pow, 2), [1, 2, 3])) == [2, 4, 8]

**Then, vary it:**

- VARIATION: swap compose order to apply right-to-left
- VARIATION: use attrgetter on dataclass instances instead of dicts

## `s4_classes.py`: Classes — iterable vs iterator, __slots__.

In [ ]:
class Window:
    __slots__ = ("data", "size")
    def __init__(self, data, size): self.data, self.size = data, size
    def __iter__(self):
        for i in range(len(self.data) - self.size + 1):
            yield self.data[i:i + self.size]
    def __len__(self): return max(0, len(self.data) - self.size + 1)
w = Window([1, 2, 3, 4], 2)

In [ ]:
assert list(w) == [[1, 2], [2, 3], [3, 4]] and len(w) == 3
assert list(w) == list(w)   # re-iterable: each iter() makes a fresh generator

In [ ]:
class Countdown:
    def __init__(self, n): self.n = n
    def __iter__(self): return self
    def __next__(self):
        if self.n <= 0: raise StopIteration
        self.n -= 1
        return self.n + 1
c = Countdown(3)

In [ ]:
assert list(c) == [3, 2, 1] and list(c) == []   # one-shot: it is its own iterator

**Then, vary it:**

- VARIATION: try adding a new attribute to Window to see __slots__ at work
- VARIATION: make Countdown re-iterable by returning a fresh iterator in __iter__

## `s5_errors.py`: Errors — csv with bad rows, StringIO as a fake file.

In [ ]:
import csv, io
data = "name,qty\na,1\nb,x\nc,3\n"
good, bad = [], []
for row in csv.DictReader(io.StringIO(data)):
    try: good.append((row["name"], int(row["qty"])))
    except ValueError: bad.append(row["name"])

In [ ]:
assert good == [("a", 1), ("c", 3)] and bad == ["b"]

In [ ]:
out = io.StringIO()
csv.writer(out).writerow(["a", 1])

In [ ]:
assert out.getvalue() == "a,1\r\n"

**Then, vary it:**

- VARIATION: collect (line_number, error) for bad rows via enumerate
- VARIATION: handle a missing column (KeyError) separately from bad ints

## `s6_stdlib.py`: Stdlib — the itertools grab-bag.

In [ ]:
from itertools import combinations, groupby, islice, permutations, product, zip_longest
assert len(list(product("ab", repeat=2))) == 4
assert list(combinations("abc", 2)) == [("a", "b"), ("a", "c"), ("b", "c")]
assert len(list(permutations(range(3)))) == 6
assert list(zip_longest([1, 2], [9], fillvalue=0)) == [(1, 9), (2, 0)]
assert list(islice(iter(range(100)), 3)) == [0, 1, 2]
assert [(k, len(list(g))) for k, g in groupby("aabccc")] == [("a", 2), ("b", 1), ("c", 3)]

**Then, vary it:**

- VARIATION: itertools.tee a generator and consume both copies
- VARIATION: build a grid of coordinates with product(range(3), repeat=2)

## `s7_concurrency.py`: Concurrency — CPU-bound fan-out with chunksize.

In [ ]:
%%writefile _pool_worker.py
def cpu(n): return sum(i * i for i in range(n))

In [ ]:
import importlib; importlib.invalidate_caches()
from concurrent.futures import ProcessPoolExecutor
from _pool_worker import cpu

with ProcessPoolExecutor(2) as ex:
    r = list(ex.map(cpu, [1000] * 4, chunksize=2))

In [ ]:
assert r == [cpu(1000)] * 4

**Then, vary it:**

Notebook note: process-pool workers must live in an importable module (spawned children can't see functions defined in cells), so the worker is written with `%%writefile`.
- VARIATION: time sequential vs pool for n=2_000_000; when does the pool win?
- VARIATION: why must `cpu` be defined at module top level (pickling)?

## `s8_quality.py`: Quality — measure, don't guess (memory + time).

In [ ]:
import sys, timeit
lst = [i for i in range(100_000)]
gen = (i for i in range(100_000))

In [ ]:
import sys, timeit

assert sys.getsizeof(gen) < sys.getsizeof(lst)

In [ ]:
a = timeit.timeit("sum(x for x in range(1000))", number=200)
b = timeit.timeit("sum(range(1000))", number=200)

In [ ]:
assert b < a

In [ ]:
print(f"genexp {a:.4f}s vs builtin {b:.4f}s")

**Then, vary it:**

- VARIATION: profile a function with cProfile.run and read the top lines
- VARIATION: use tracemalloc to compare list vs generator peak memory

## `kata.py`: KATA — streaming csv -> filter -> group-sum pipeline.

In [ ]:
import csv, io
from itertools import groupby
from operator import itemgetter
RAW = "cat,amt\nfood,5\nfun,3\nfood,7\nbad,x\nfun,1\n"
def rows(f):
    for r in csv.DictReader(f):
        try: yield r["cat"], int(r["amt"])
        except ValueError: continue
def totals(pairs):
    for k, g in groupby(sorted(pairs), key=itemgetter(0)):
        yield k, sum(a for _, a in g)

In [ ]:
import csv, io

assert dict(totals(rows(io.StringIO(RAW)))) == {"food": 12, "fun": 4}

**Then, vary it:**

- KATA: rewrite totals with collections.Counter; which is clearer?